# 00 — Start a PTA release project

**You will:** identify the PLEB code being run, generate two artificial pulsars,
initialise an isolated dataset repository, and declare the scope of your release.

Launch `./learn.sh release /path/to/release-learning --port 8889`, open this notebook,
and run each cell with Shift+Enter. Continue in numerical order. This course is
independent of the timing/statistics lessons. Every PLEB command and config is shown
in the notebooks; `release_tools` supplies only simulation and file/log utilities.

The image supplies timing software and dependencies. **This course runs the PLEB
source mounted from your checkout**, so it includes the current comment-only duplicate
and reviewed-keep fixes. We record source hashes. The earlier timing course still uses
the image's installed PLEB. Do not confuse a source revision with the image revision.

You are building a **synthetic training release**, not reproducing EPTA DR3.
It has two pulsars, two observing systems, white noise, four duplicate occurrences
and four deliberately displaced observations. Starting timing models are known and
include one backend jump with A as the phase reference. Real releases need independently
approved starting models and telescope clock files.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
import pleb, importlib.metadata
print('PLEB source:', pleb.__file__)
assert str(Path(pleb.__file__).resolve()).startswith('/pleb-source/'), 'Use the release launcher'
software = {name:importlib.metadata.version(name) for name in ['libstempo','numpy','scipy','pandas']}
software['pleb_source_sha256'] = {n:h for n,h in lab.hashes(Path(pleb.__file__).parent).items() if n.endswith('.py')}
if (ROOT/'runtime/container.sha256').exists():
    software['container_sha256'] = (ROOT/'runtime/container.sha256').read_text().split()[0]
(ROOT/'software.json').write_text(json.dumps(software,indent=2))
print('Timing runtime:', os.environ['TEMPO2'])

## Establish the input boundary

`raw/` represents files delivered by data providers. Do not edit it during cleaning.
`dataset-repo/data/` will be the canonical dataset tracked by Git.
`configs/`, `results/`, `review/`, and `exports/` serve different roles.
Inspect the raw TIM rows: FORMAT 1, filename, frequency (MHz), MJD, uncertainty (µs),
observatory, and flag/value pairs. `-sys A` and `-sys B` identify the observing systems.

In [ ]:
raw_manifest = lab.prepare_raw(ROOT)
print('Synthetic pulsars:', raw_manifest['pulsars'])
print('Expected physical TOA rows:', raw_manifest['expected_physical_rows'])
print((ROOT/'raw/A'/f'{PULSARS[0]}.tim').read_text().splitlines()[:4])
display(pd.DataFrame([{'file':n,'sha256':h} for n,h in raw_manifest['sha256'].items()]))

## Initialise the dataset repository

All following Git commands operate on this new repository. `scaffold` is an empty
starting point; `raw_ingest`, `prepared`, `reviewed`, and `release_model` will record
progressively reviewed products. Results are stored outside that Git worktree.
Branches are evidence of what changed, not approval by themselves.

In [ ]:
REPO.mkdir(exist_ok=True)
if not (REPO/'.git').exists():
    lab.command(['git','init','-b','scaffold',REPO])
    lab.command(['git','-C',REPO,'config','user.name','PLEB synthetic training'])
    lab.command(['git','-C',REPO,'config','user.email','training@example.invalid'])
    (REPO/'README.md').write_text('Synthetic PTA release training only.\n')
    (REPO/'.gitignore').write_text('logs/\ndata/ingest_reports/run_settings/\n')
    lab.command(['git','-C',REPO,'add','README.md','.gitignore'])
    lab.command(['git','-C',REPO,'commit','-m','Initialise training dataset repository'])
lab.command(['git','-C',REPO,'status','--short','--branch'])

## Declare stage defaults explicitly

`home_dir` is the dataset Git repository, `dataset_name` is its relative data path,
and `results_dir` is outside it. We are already inside the image, so
`tempo2_native=true` avoids nested container launches. These are environment choices,
not universal observing-system policies. We will write separate runnable TOMLs for each stage.

In [ ]:
common = dict(home_dir=str(REPO), dataset_name='data', results_dir=str(ROOT/'results'),
    singularity_image='/unused-inside-container.sif', tempo2_native=True,
    pulsars=PULSARS, jobs=1, epoch='58730', run_tempo2=False, run_pqc=False,
    run_fix_dataset=False, qc_report=False, make_plots=False, make_reports=False,
    make_covmat=False, consolidated_report=False, cleanup_output_tree=False,
    cleanup_work_dir=False, force_rerun=True)
(CONFIGS/'common.json').write_text(json.dumps(common,indent=2))
display(pd.Series(common))

**Your task:** draw the raw tree and canonical tree. Explain why editing a raw TIM
would break the input manifest. Verify a raw checksum yourself. Next: write the ingest mapping.